<a href="https://colab.research.google.com/github/vijaysingh60616-sudo/vijaysingh60616-sudo.github.io/blob/main/day11%2612.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 11 and 12

# step 1 . Mean od Centering

$$X_{centered} = X - \mu$$


# step 2. Convariance Matrix
$$C = \frac{1}{n-1} X_{centered}^T X_{centered}$$

n= number of sample
C = sematrix matrix

# step 3.Eigendecomposition

$$C v = \lambda v$$Eigenvectors ($v$):  (Principal Components)
Eigenvalues ($\lambda$): direction variance

# Step 4. PRojection

$$X_{pca} = X_{centered} \cdot W$$




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class PCA_From_Scratch:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None
        self.mean = None
        self.eigenvalues = None

    def fit(self, X):
        # Step 1: Mean Centering
        self.mean = np.mean(X, axis=0)
        X_centered = X - self.mean

        # Step 2: Covariance Matrix
        # rowvar=False means each column is a variable
        cov_matrix = np.cov(X_centered, rowvar=False)

        # Step 3: Eigendecomposition
        # np.linalg.eigh is used for symmetric matrices (like covariance matrices)
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

        # Step 4: Sort eigenvectors by eigenvalues in descending order
        # eigh returns eigenvalues in ascending order, so we reverse it [::-1]
        sorted_indices = np.argsort(eigenvalues)[::-1]
        self.eigenvalues = eigenvalues[sorted_indices]
        eigenvectors = eigenvectors[:, sorted_indices]

        # Store the first 'n_components' eigenvectors (Principal Components)
        self.components = eigenvectors[:, :self.n_components]

    def transform(self, X):
        # Project the data onto the principal components
        X_centered = X - self.mean
        return np.dot(X_centered, self.components)

# 1. Create a correlated dataset (2D)
np.random.seed(42)
X_dummy = np.dot(np.random.rand(2, 2), np.random.randn(2, 200)).T

# 2. Initialize and apply our PCA
pca = PCA_From_Scratch(n_components=1) # Reducing from 2D to 1D
pca.fit(X_dummy)
X_projected = pca.transform(X_dummy)

print(f"Original Shape: {X_dummy.shape}")
print(f"Projected Shape: {X_projected.shape}")
print(f"Top Eigenvector (Direction): {pca.components.flatten()}")

# 3. Visualize
plt.figure(figsize=(8, 4))
plt.scatter(X_dummy[:, 0], X_dummy[:, 1], alpha=0.5, label='Original Data')
# Reconstruct data to 2D for visualization of the axis
X_reconstructed = np.dot(X_projected, pca.components.T) + pca.mean
plt.scatter(X_reconstructed[:, 0], X_reconstructed[:, 1], color='red', alpha=0.8, label='Projected Data (1D representation)')
plt.title("PCA from Scratch: 2D to 1D Projection")
plt.legend()
plt.show()

# Singular Value Decompostion

$$X_{centered} = U \Sigma V^T$$$U$: Left singular vectors
$\Sigma$: Singular value
$V^T$: Right singular vectors


In [ ]:
def pca_using_svd(X, n_components):
  X = X - np.mean(X, axis=0)
  U, S, Vt = np.linalg.svd(X, full_matrices=False)
  components = Vt[:,n_components]
  return np.dot(X, components)

X_pca_svd = pca_using_svd(X_dummy, n_components=1)



# New Section

$$p_{j\vert{}i} = \frac{\exp(-\Vert{}x_i - x_j\Vert{}^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\Vert{}x_i - x_k\Vert{}^2 / 2\sigma_i^2)}$$


$$q_{j\vert{}i} = \frac{(1 + \Vert{}y_i - y_j\Vert{}^2)^{-1}}{\sum_{k \neq l} (1 + \Vert{}y_k - y_l\Vert{}^2)^{-1}}$$



$$KL(P \vert{}\vert{} Q) = \sum_i \sum_j p_{j\vert{}i} \log \frac{p_{j\vert{}i}}{q_{j\vert{}i}}$$




$$p_{j\vert{}i} = \exp \left( - \frac{\max(0, d(x_i, x_j) - \rho_i)}{\sigma_i} \right)$$




$$C_{fuzzy} = \sum_{i,j} \left[ p_{j\vert{}i} \log \frac{p_{j\vert{}i}}{q_{j\vert{}i}} + (1 - p_{j\vert{}i}) \log \frac{1 - p_{j\vert{}i}}{1 - q_{j\vert{}i}} \right]$$


In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
import umap.umap_ as umap

digits = load_digits()
X, y =digits.data, digits.target

print(f"Original Shape: {X.shape}")

In [ ]:
print("Running t-SNE ...")
tsne = TSNE(n_components=2,perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X)
print(f"TSNE Projection Shape: {X_tsne.shape}")


In [ ]:
print("running UMAP ...")
umap_reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='euclidean')
X_umap = umap_reducer.fit_transform(X)
print(f"UMAP Projection Shape: {X_umap.shape}")
#

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# t-SNE Plot
scatter1 = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='tab10', alpha=0.7, s=15)
axes[0].set_title("t-SNE Manifold Learning")
axes[0].axis('off')

# UMAP Plot
scatter2 = axes[1].scatter(X_umap[:, 0], X_umap[:, 1], c=y, cmap='tab10', alpha=0.7, s=15)
axes[1].set_title("UMAP Manifold Learning")
axes[1].axis('off')

# Colorbar add karna digits (0-9) ke liye
fig.colorbar(scatter1, ax=axes, orientation='horizontal', label='Digit Classes (0-9)')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(10, 10, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X[i].reshape(8, 8), cmap='gray')
    ax.set_title(f'Digit {y[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()
